[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLGlobalHealth/cntmosaic/blob/main/docs/source/tutorials/02_Tutorial_SocialMix.ipynb)

# Tutorial: Estimating Social Contact Matrices with SocialMix

## Introduction

Social contact surveys ask respondents to record the contacts they had on a given day, together with characteristics of each contact such as their age. A contact matrix summarises these data: an age-stratified contact matrix gives the *average* number of contacts that a person in one age group has with people in another age group. Contact matrices are a standard input to age-structured models of infectious disease transmission.

`cntmosaic` provides a Python implementation of the R package [socialmixr](https://epiforecasts.io/socialmixr/). The `SocialMix` class estimates contact matrices from survey data in the same way as `socialmixr`. This tutorial follows the structure of the socialmixr [introductory vignette](https://epiforecasts.io/socialmixr/articles/socialmixr.html) and reproduces its examples with cntmosaic. Some of the results are close to, but not always identical with, those of socialmixr due to some differences in the details. Throughout, we use the UK branch of the POLYMOD survey (Mossong et al., 2008) as the example.

## Setup

In [ ]:
# @title Setup (Google Colab only)
# Installs cntmosaic from GitHub when the notebook runs in Google Colab.
# Does nothing when run locally.
import subprocess
import sys

try:
    import google.colab  # noqa: F401
except ImportError:
    pass
else:
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--quiet",
            "git+https://github.com/MLGlobalHealth/cntmosaic.git",
        ]
    )

In [ ]:
import warnings

import numpy as np
import pandas as pd

from cntmosaic.datasets import load_polymod_uk
from cntmosaic.utils import AgeGroupSpecs
from cntmosaic.dataloader import ContactData, ParticipantData, PopulationData
from cntmosaic.models import SocialMix
from cntmosaic.analysis import ModelSummariserSocialMix
from cntmosaic.vis import plot_mosaic_pixilated

# Visualisation
import altair as alt

`cntmosaic` ships the POLYMOD UK data with the package. The data can be loaded via a convenient utility function `load_polymod_uk` which is part of the `datasets` module. This function load a Python dictionary with three tables:

1. the **respondents**, one row per respondent, with their age and gender;
2. the **contacts**, one row per reported contact, with the age and gender of the contact and the setting of the contact;
3. the **population** of the UK in 2011 by single year of age and sex.

In [ ]:
# Load and unpack the POLYMOD UK survey data
survey_data = load_polymod_uk()
df_part = survey_data["participants"]
df_cnt = survey_data["contacts"]
df_pop = survey_data["population"]

# ===== Respondent data =====
df_part["part_gender"] = pd.Categorical(df_part["part_gender"], categories=["M", "F"])

# ===== Contact data =====
df_cnt = df_cnt.rename(columns={"cnt_age_exact": "cnt_age"})
df_cnt["cnt_gender"] = pd.Categorical(df_cnt["cnt_gender"], categories=["M", "F"])

# ===== Population data =====
df_pop["age"] = df_pop["age"].astype(str).str.replace("+", "").astype(int)
df_pop = df_pop[df_pop["age"] <= 79]
df_pop["gender"] = pd.Categorical(df_pop["gender"], categories=["M", "F"])

# Population by age only (summed over gender)
df_pop_age = df_pop.groupby("age", as_index=False)["P"].sum()

print(f"{len(df_part)} respondents, {len(df_cnt)} contacts")

## 1. Estimating a contact matrix

Let $\mathcal{N}_c$ be the set of respondents in age group $c$, and let $y_{id}$ be the number of contacts that respondent $i$ reported with people in age group $d$. The **contact intensity** from age group $c$ to age group $d$ is

$$
m_{cd} = \frac{1}{|\mathcal{N}_c|} \sum_{i \in \mathcal{N}_c} y_{id},
$$

that is, the average number of contacts that a respondent of age group $c$ has with members of age group $d$ on a survey day. Respondents who reported no contacts are included in $\mathcal{N}_c$ with $y_{id} = 0$.

In `cntmosaic` the estimation takes three steps:

1. wrap each data frame in a data container (`ParticipantData`, `ContactData`, and later `PopulationData`), which validates the data and standardises the column names;
2. define the age groups with `AgeGroupSpecs`;
3. pass both to `SocialMix`.

In [ ]:
age_group_specs = AgeGroupSpecs(min=0, max=79, step=5)

part_data = ParticipantData(df_part, id_col="part_id", age_col="part_age")
cnt_data = ContactData(df_cnt, id_col="part_id", age_col="cnt_age")

sm = SocialMix(part_data, cnt_data, age_group_specs)

`ContactData` drops the 3 contacts with a missing age and reports this in a warning.

The method `cint()` returns the contact intensity matrices as a dictionary. Its keys have the form `"source->target"`, where source and target are the strata of respondents and contacts. Without stratification there is a single matrix, `"All->All"`. Each value is a `ContactSummary`, whose attribute `central` holds the point estimate. Rows are respondent age groups $c$ and columns are contact age groups $d$.

In [ ]:
m = sm.cint()["All->All"].central

age_labels = [f"{l}-{r}" for l, r in zip(age_group_specs.left, age_group_specs.right)]
pd.DataFrame(m, index=age_labels, columns=age_labels).round(2)

### Assigning age groups

`AgeGroupSpecs(0, 79, 5)` defines 16 age groups of 5 years, $[0, 4], [5, 9], \ldots, [75, 79]$, with inclusive bounds. Groups of unequal width are defined with `cuts`, e.g. `AgeGroupSpecs(0, 79, cuts=[1, 5, 15])` for the groups $[0, 0], [1, 4], [5, 14], [15, 79]$ used in the socialmixr vignette, or with explicit bounds, `AgeGroupSpecs(age_min=[...], age_max=[...])`.

When the containers are given single-year ages (`age_col=`), `SocialMix` assigns the age groups itself. It works on copies of the containers, so the containers we passed in are not modified. The copies are available as `sm.part_data` and `sm.cnt_data`. The respondent identifier is renamed to `id`, and the contact table gains a column `y` with the number of contacts in each row (here always 1).

In [ ]:
sm.part_data.data.head()

In [ ]:
sm.cnt_data.data.head()

Respondents whose age falls outside the age groups are dropped with a warning. Contacts outside the age groups are not counted: here the 103 contacts aged 80 or over do not appear in $m_{cd}$. If the ages are already grouped, we pass the group column with `age_grp_col=` instead, and the groups must agree with `age_group_specs`.

## 2. Bootstrapping

The point estimate $m_{cd}$ does not convey its sampling uncertainty. A simple way to estimate the uncertainty is the bootstrap:

1. sample respondents with replacement, keeping all the contacts of each sampled respondent;
2. recompute $m_{cd}$ on the resample;
3. repeat many times, and summarise the resulting matrices by their mean, standard deviation or quantiles.

The socialmixr vignette writes a helper function for the resampling. In cntmosaic it is built into `SocialMix` as `run_inference_bootstrap()`.

In [ ]:
boot = sm.run_inference_bootstrap(n_boot=1000, random_state=1)

Before resampling, `SocialMix` checks that every respondent age group has enough respondents for a resample to be very unlikely to leave it empty. Otherwise, it merges the smallest group with a neighbouring group and issues a warning. Here the groups 70–74 and 75–79 were merged into 70–79. The merge persists after the bootstrap: `sm.age_group_specs` and the point estimates are updated to the merged groups.

In [ ]:
print("left: ", sm.age_group_specs.left)
print("right:", sm.age_group_specs.right)

The returned `BootstrapResults` object summarises the bootstrap matrices with `mean()`, `std()` and `quantiles()`. Each returns a dictionary with the same keys as `cint()`.

In [ ]:
m_mean = boot.mean()["All->All"]
m_lower, m_upper = boot.quantiles(q=[0.025, 0.975])["All->All"]

# Contacts of 10-14 year olds with 10-14 year olds (third age group)
c = d = 2
print(f"Point estimate:         {sm.cint()['All->All'].central[c, d]:.2f}")
print(f"Bootstrap mean:         {m_mean[c, d]:.2f}")
print(f"95% bootstrap interval: [{m_lower[c, d]:.2f}, {m_upper[c, d]:.2f}]")

`ModelSummariserSocialMix` collects the bootstrap median and the 95% bootstrap interval of every entry in a `ContactSummary` (attributes `central`, `lower` and `upper`), which can be passed directly to the plotting functions. Below we plot the width of the interval.

In [ ]:
summ_cint = ModelSummariserSocialMix(sm).summarise_cint()

plot_mosaic_pixilated(
    summ_cint["All->All"].upper - summ_cint["All->All"].lower,
    summ_cint["All->All"].age_group_specs,
    title="Width of 95% bootstrap interval",
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="Width",
    color_scheme="viridis",
)

## 3. Symmetric contact matrices

Let $P_c$ be the population size of age group $c$. The total number of contacts between age groups $c$ and $d$ in the population is $m_{cd} P_c$ when counted from the side of age group $c$, and $m_{dc} P_d$ when counted from the side of age group $d$. Since every contact between a member of $c$ and a member of $d$ is the same contact seen from either side, the two totals should agree:

$$
m_{cd} P_c = m_{dc} P_d.
$$

In survey estimates this rarely holds exactly, because of sampling variation and reporting errors. The reciprocity adjustment replaces $m_{cd}$ with the average of the two totals, divided by $P_c$:

$$
\tilde{m}_{cd} = \frac{m_{cd} P_c + m_{dc} P_d}{2 P_c}.
$$

This requires population data. We wrap the population table in `PopulationData` and set `apply_reciprocity=True`. The adjustment is off by default, because whether it is appropriate depends on the analysis.

In [ ]:
pop_data = PopulationData(df_pop_age, age_col="age", size_col="P")

sm_sym = SocialMix(
    part_data, cnt_data, age_group_specs, pop_data=pop_data, apply_reciprocity=True
)

The population table has single-year ages, so `SocialMix` sums it into the age groups and warns that it has done so. We silence this warning for the rest of the tutorial.

We now compare the total contacts $m_{cd} P_c$ before and after the adjustment.

In [ ]:
warnings.filterwarnings("ignore", message="Population data is provided in 1-year")

sm_raw = SocialMix(part_data, cnt_data, age_group_specs, pop_data=pop_data)
P = sm_sym.P

total_raw = sm_raw.cint()["All->All"].central * P[:, np.newaxis]
total_sym = sm_sym.cint()["All->All"].central * P[:, np.newaxis]

print(
    "Total contacts symmetric without adjustment:", np.allclose(total_raw, total_raw.T)
)
print(
    "Total contacts symmetric with adjustment:   ", np.allclose(total_sym, total_sym.T)
)

## 4. Contact rates

The contact intensity $m_{cd}$ depends on how many people there are in age group $d$: a larger age group offers more people to meet. Transmission models usually require the **contact rate**

$$
\gamma_{cd} = \frac{m_{cd}}{P_d},
$$

the average number of contacts that a member of age group $c$ has with one particular member of age group $d$. The method `rate()` returns $\gamma_{cd}$ in the same format as `cint()`, and requires population data.

Recall that the reciprocity adjustment ensures $m_{cd} P_c = m_{dc} P_d$. Dividing both sides by $P_c P_d$ gives $\gamma_{cd} = \gamma_{dc}$, so with the adjustment the contact rate matrix is symmetric.

In [ ]:
gamma = sm_sym.rate()["All->All"].central

print("Contact rate matrix symmetric:", np.allclose(gamma, gamma.T))

plot_mosaic_pixilated(
    sm_sym.rate()["All->All"],
    title="Contact rate",
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="Rate",
)

## 5. Proportionate-mixing factorisation

The socialmixr function `split_matrix()` factorises a contact intensity matrix into a few interpretable parts. In cntmosaic this is `prop_mix_factorisation()`. Let $m_c = \sum_d m_{cd}$ be the **marginal contact intensity** of age group $c$, that is, the average total number of contacts of a respondent of age group $c$, and let $p_d = P_d / \sum_{d'} P_{d'}$ be the population share of age group $d$. The factorisation is

$$
m_{cd} = m_c \, a_{cd} \, p_d,
$$

where $a_{cd} = m_{cd} / (m_c \, p_d)$ is the **assortativity**: the share of age group $c$'s contacts that are with age group $d$, divided by the population share $p_d$. If $a_{cd} = 1$ for all $d$, age group $c$ distributes its contacts in proportion to the size of each age group (proportionate mixing). A diagonal entry $a_{cc} > 1$ means that age group $c$ mixes more with its own age group than proportionate mixing would imply.

socialmixr further splits the marginal contact intensity as

$$
m_c = \bar{m} \, q \, r_c,
$$

where

1. $\bar{m} = \sum_c p_c \, m_c$ is the population-weighted mean of the marginal contact intensities;
2. $\bar{m} q$ is the largest eigenvalue of the contact intensity matrix with entries $m_{cd}$. It equals the basic reproduction number $R_0$ if every contact results in transmission;
3. $r_c = m_c / (\bar{m} q)$ is the marginal contact intensity relative to $\bar{m} q$.

The factorisation requires population data and is defined for matrices without stratification.

In [ ]:
fac = sm_sym.prop_mix_factorisation()

print(f"Mean contacts (m-bar):        {fac.mean_contacts:.2f}")
print(f"Normalisation (q):            {fac.normalisation:.3f}")
print(f"Largest eigenvalue (m-bar q): {fac.spectral_radius:.2f}")
print(
    "Factorisation reproduces the matrix:", np.allclose(fac.reconstruct(), fac.matrix)
)

The attributes correspond to the output of `split_matrix()` as follows.

| socialmixr       | cntmosaic           | symbol      |
|------------------|---------------------|-------------|
| `$matrix`        | `assortativity`     | $a_{cd}$    |
| `$mean.contacts` | `mean_contacts`     | $\bar{m}$   |
| `$normalisation` | `normalisation`     | $q$         |
| `$contacts`      | `relative_contacts` | $r_c$       |
| (not returned)   | `contacts`          | $m_c$       |
| (not returned)   | `spectral_radius`   | $\bar{m} q$ |
| (population)     | `population_share`  | $p_d$       |

In [ ]:
plot_mosaic_pixilated(
    fac.assortativity,
    fac.age_group_specs,
    title="Assortativity",
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="a",
    color_scheme="redblue",
    color_mid=1,  # proportionate mixing
)

## 6. Filtering

Often we are interested in a subset of contacts, for example those that took place at school. In `cntmosaic` we filter the data frames with pandas before wrapping them in the containers.

Filtering the contacts keeps all respondents. The resulting $m_{cd}$ is then the average number of contacts *of the selected kind*, and respondents who reported no such contacts count with $y_{id} = 0$. Filtering the respondents instead, e.g. to respondents who reported on a weekday, changes the set $\mathcal{N}_c$ over which we average.

We reproduce the three examples of the vignette:

1. contacts at school;
2. contacts at work that involved physical contact (`phys_contact == 1`);
3. contacts at home with males that lasted four hours or more (`duration_multi == 5`).

In [ ]:
filters = {
    "School": df_cnt["cnt_school"],
    "Work, physical": df_cnt["cnt_work"] & (df_cnt["phys_contact"] == 1),
    "Home, male, 4h+": df_cnt["cnt_home"]
    & (df_cnt["cnt_gender"] == "M")
    & (df_cnt["duration_multi"] == 5),
}

charts = []
for i, (name, mask) in enumerate(filters.items()):
    cnt_data_sub = ContactData(df_cnt[mask], id_col="part_id", age_col="cnt_age")
    sm_sub = SocialMix(part_data, cnt_data_sub, age_group_specs)
    charts.append(
        plot_mosaic_pixilated(
            sm_sub.cint()["All->All"],
            title=name,
            xlabel="Age of respondent",
            ylabel="Age of contact",
            zlabel="Intensity",
            show_y_label=(i == 0),
            show_y_ticks=(i == 0),
            width=200,
            height=200,
        )
    )

alt.hconcat(*charts).resolve_scale(color="independent")

## 7. Stratification: generalised contact matrices

Filtering gives one matrix per subset of the data. Instead, we can stratify respondents and contacts by further characteristics, such as gender, and estimate all the combinations at once. Manna et al. (2024) call these **generalised contact matrices**. Let $k$ and $l$ be strata of respondents and contacts. The contact intensity $m^{kl}_{cd}$ is the average number of contacts that a member of age group $c$ in stratum $k$ has with members of age group $d$ in stratum $l$.

We specify the stratification variables with `strat_var_cols=` in each container. The variable carries the prefix `part_` in the respondent table and `cnt_` in the contact table, and no prefix in the population table. When the same variable (here `gender`) appears on both sides, `SocialMix` estimates a matrix for every combination of strata. The keys of `cint()` are then `"M->M"`, `"M->F"`, `"F->M"` and `"F->F"`, where the first stratum is that of the respondent and the second that of the contact.

For the reciprocity adjustment and the contact rates, the population must be stratified in the same way. The adjustment then ensures $m^{kl}_{cd} P^k_c = m^{lk}_{dc} P^l_d$.

In [ ]:
part_data_g = ParticipantData(
    df_part, id_col="part_id", age_col="part_age", strat_var_cols="part_gender"
)
cnt_data_g = ContactData(
    df_cnt, id_col="part_id", age_col="cnt_age", strat_var_cols="cnt_gender"
)
pop_data_g = PopulationData(
    df_pop, age_col="age", size_col="P", strat_var_cols="gender"
)

sm_g = SocialMix(
    part_data_g,
    cnt_data_g,
    age_group_specs,
    pop_data=pop_data_g,
    apply_reciprocity=True,
)

cint_g = sm_g.cint()
list(cint_g.keys())

`ContactData` drops the contacts with a missing gender, since they cannot be assigned to a stratum. We plot the four blocks on a common colour scale.

In [ ]:
vmax = max(cs.central.max() for cs in cint_g.values())
blocks = {
    key: plot_mosaic_pixilated(
        cs,
        title=key,
        xlabel="Age of respondent",
        ylabel="Age of contact",
        zlabel="Intensity",
        color_min=0,
        color_max=vmax,
        width=200,
        height=200,
        show_y_label=(i % 2 == 0),  # left column
        show_y_ticks=(i % 2 == 0),
        show_x_label=(i >= 2),  # bottom row
        show_x_ticks=(i >= 2),
    )
    for i, (key, cs) in enumerate(cint_g.items())
}

(blocks["M->M"] | blocks["M->F"]) & (blocks["F->M"] | blocks["F->F"])

If only the respondents are stratified, the matrices are keyed `"M->All"` and `"F->All"`: one matrix per respondent stratum, with contacts of any gender. In this case the reciprocity adjustment is not applied, since the contacts are not stratified in the same way as the respondents.

In [ ]:
sm_part_g = SocialMix(part_data_g, cnt_data, age_group_specs)
list(sm_part_g.cint().keys())

## 8. Respondent weights

A contact matrix is an *average* over the respondents of each age group. If the composition of the sample differs from that of the population in a factor that influences contact behaviour, it would be prudent to weight the respondents so that the average reflects the population. For instance, contacts on weekdays, when children go to school and adults go to work, differ from those at weekends. If weekday and weekend reports are not in the ratio 5:2, a naive average could be biased towards one or the other.

With respondent weights $w_i$, the contact intensity becomes

$$
m_{cd} = \frac{\sum_{i \in \mathcal{N}_c} w_i \, y_{id}}{\sum_{i \in \mathcal{N}_c} w_i}.
$$

Weighting is opt-in. We describe the weights with a `WeightSpecs` object, built from one or more weight factors, and pass it to `SocialMix` with `weight_specs=`. As in the vignette, we weight by day of week (`DayOfWeekWeights`, 5:2 within each age group) and by single-year age within each age group (`AgePostStratWeights`), and truncate large weights with `threshold=3`. The weights have mean 1 in each age group, so weights above 3 are set to 3, and the weights are then renormalised. The day of week is stored in the column `dayofweek` (0 = Sunday, ..., 6 = Saturday) and is passed to `ParticipantData` with `aux_cols=`, which keeps the column for weighting without treating it as a stratification variable.

In [ ]:
from cntmosaic.weights import AgePostStratWeights, DayOfWeekWeights, WeightSpecs

part_data_w = ParticipantData(
    df_part, id_col="part_id", age_col="part_age", aux_cols="dayofweek"
)
weight_specs = WeightSpecs(
    DayOfWeekWeights(), AgePostStratWeights(pop_data), threshold=3
)

sm_w = SocialMix(
    part_data_w, cnt_data, age_group_specs, pop_data=pop_data, weight_specs=weight_specs
)
sm_w.weights

`cint()` now returns the weighted estimate. The unweighted estimate of the same fit is available with `cint(weighted=False)`, and `sm_w.weights` holds diagnostics such as the effective sample size in each age group.

In [ ]:
m_w = sm_w.cint()["All->All"]
m_u = sm_w.cint(weighted=False)["All->All"]

(
    plot_mosaic_pixilated(
        m_u,
        title="Unweighted",
        xlabel="Age of respondent",
        ylabel="Age of contact",
        zlabel="Intensity",
        width=200,
        height=200,
    )
    | plot_mosaic_pixilated(
        m_w,
        title="Weighted",
        xlabel="Age of respondent",
        ylabel="Age of contact",
        zlabel="Intensity",
        width=200,
        height=200,
    )
)

When to weight, how each factor computes its weights, and how weights interact with the bootstrap are covered in detail in [Tutorial 05: Survey Weights in SocialMix](05_Tutorial_SocialMix_Weights.ipynb).

## 9. Plotting

`plot_mosaic_pixilated()` plots a matrix whose age groups may have different widths. Each age group is drawn with a width proportional to the number of years it spans. It accepts either a `ContactSummary`, which carries its own age groups, or a NumPy array together with an `AgeGroupSpecs`. The x axis shows the age group of the respondent and the y axis that of the contact. The main arguments are the labels (`title`, `xlabel`, `ylabel`, `zlabel`), the size (`width`, `height`) and the colour scale (`color_scheme`, `color_reverse`, `color_min`, `color_max`, `color_mid`).

In [ ]:
plot_mosaic_pixilated(
    m_mean,
    sm.age_group_specs,
    title="Bootstrap mean contact intensity",
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="Intensity",
)

The function returns an Altair chart, so it can be further customised and combined with Altair's own methods: `.properties()` for the size and title, `|` and `&` to place charts side by side or on top of each other (as in the sections above), `.configure_*()` for the overall appearance, and `.save()` to write the chart to a file.

In [ ]:
chart = plot_mosaic_pixilated(
    m_mean,
    sm.age_group_specs,
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="Intensity",
)

chart.properties(width=350, height=350, title="POLYMOD UK").configure_axis(
    labelFontSize=11, titleFontSize=13
)

## References

- Funk, S. et al. *socialmixr: Social Mixing Matrices for Infectious Disease Modelling.* R package. Introductory vignette: <https://epiforecasts.io/socialmixr/articles/socialmixr.html>.
- Manna, A., Dall'Amico, L., Tizzoni, M., Karsai, M. and Perra, N. (2024). Generalized contact matrices allow integrating socioeconomic variables into epidemic models. *Science Advances*, 10(41), eadk4606.
- Mossong, J. et al. (2008). Social contacts and mixing patterns relevant to the spread of infectious diseases. *PLoS Medicine*, 5(3), e74.